# LILY WAN 2.2 STUDIO

In Kaggle **Settings**, enable **Internet** and choose **GPU T4 ×2** (preferred) or **P100**. Run the single cell below, wait for **[5/5] READY**, then open its share link. Upload an image, enter a motion prompt, and start with **⚡ TURBO**.

This is a complete replacement using native Wan 2.2 5B inference, with automatic P100 settings. GPU 0 generates; GPU 1 is intentionally unused. Initial model download: **16.9 GiB**, plus **1.5 GiB** free-space reserve. CPU offloading requires **21 GiB available system RAM**. Keep your Kaggle session running while using the UI.

**Validation limit:** CPU/API, encoding, and recovery checks are possible here; full generation, peak VRAM, speed, and Kaggle share tunnels still require real Kaggle hardware. Some Kaggle Torch builds cannot execute on P100; the cell detects that before downloading weights and preserves Torch/CUDA.

[Instructions and troubleshooting](https://github.com/benruiz1024-ops/hi/blob/main/KAGGLE_WAN_README.md)


In [ ]:
# JSON literal compatibility guard
null = None
true = True
false = False

# One architecture: native ComfyUI inference, called in this Python process.
# No ComfyUI server, workflow JSON, custom-node initialization, or old notebooks.
# The official Comfy-Org weights occupy 16.9 GiB; the official Diffusers
# snapshot occupies about 32 GiB. This pinned core supports the official
# Wan22ImageToVideoLatent node and ordinary CUDA operations on legacy GPUs.
import gc
import hashlib
import importlib
import importlib.metadata as metadata
import inspect
import json
import logging
import math
import os
from pathlib import Path
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
from types import SimpleNamespace

ROOT = Path('/kaggle/working/lily_wan22_studio')
MODELS = ROOT / 'models'
OUTPUTS = ROOT / 'outputs'
TEMP = ROOT / 'temporary'
CODE = ROOT / 'ComfyUI'
GIB = 1024 ** 3
COMFY_REVISION = '72212fef660bcd7d9702fa52011d089c027a64d8'
MODEL_REPO = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
MODEL_REVISION = 'c4f60d30c55a624e35427060fdd217579a6c1d77'
MODEL_FILES = [
    {'path': 'split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',
     'size': 9999658848, 'sha256': '456f901338bd9eadbded3828b819109a9b68e8a525ca5cf8d0049a69fcfeca1e'},
    {'path': 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
     'size': 6735906897, 'sha256': 'c3355d30191f1f066b26d93fba017ae9809dce6c627dda5f6a66eaa651204f68'},
    {'path': 'split_files/vae/wan2.2_vae.safetensors',
     'size': 1409400960, 'sha256': 'e40321bd36b9709991dae2530eb4ac303dd168276980d3e9bc4b6e2b75fed156'},
]
FPS = 24
PRESET_NAMES = ['⚡ TURBO', '✨ NORMAL', '👑 MAX']
DEFAULT_NEGATIVE = 'blurry, distorted, frozen motion, text, watermark, low quality'

# Keep the loaded models and the lock across cell reruns. Never run two jobs.
if '_lily_state' not in globals():
    _lily_state = SimpleNamespace(lock=threading.Lock(), pipeline=None, demo=None,
                                 backend=None, hardware=None, prompt_cache=None)
STATE = _lily_state


class StudioError(RuntimeError):
    def __init__(self, category, detail):
        super().__init__(f'FAILED: {category}\n{detail}')


def brief(exc):
    return f'{type(exc).__name__}: {str(exc)[-1200:]}'


def run_command(command, category='ENVIRONMENT', timeout=300, cwd=None):
    # Argument arrays only: no shell, generated Python, or command substitution.
    command = [str(arg) for arg in command]
    log_path = TEMP / 'command.log'
    try:
        with log_path.open('w') as log:
            result = subprocess.run(command, cwd=cwd, stdout=log,
                                    stderr=subprocess.STDOUT, timeout=timeout)
        if result.returncode:
            tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-12:])
            raise StudioError(category, tail)
    except (OSError, subprocess.TimeoutExpired) as exc:
        raise StudioError(category, brief(exc)) from None


def check_disk(required, path=None):
    disk = shutil.disk_usage(path or ROOT)
    if disk.free < required:
        raise StudioError('INSUFFICIENT DISK SPACE',
                          f'Required: {required / GIB:.2f} GiB additional\n'
                          f'Available: {disk.free / GIB:.2f} GiB\n'
                          'Free space or start a fresh Kaggle session, then Run again.')
    return disk


def prepare_directories():
    if not ROOT.parent.is_dir():
        raise StudioError('ENVIRONMENT', 'Run this notebook inside Kaggle; /kaggle/working is unavailable.')
    check_disk(10 * 1024 ** 2, path=ROOT.parent)
    for folder in (ROOT, MODELS, OUTPUTS, TEMP):
        folder.mkdir(parents=True, exist_ok=True)
        probe = folder / '.write_probe'
        probe.write_text('ok')
        probe.unlink()
    # Only abandoned encodes created here. Preserve every finished MP4 and all
    # resumable model .part files; never purge shared pip/Hugging Face caches.
    for path in TEMP.glob('encode_*.part.mp4'):
        path.unlink()
    for name in ('lily_pkgs', 'lily_comfy_env', 'ComfyUI', 'Practical-RIFE'):
        old = ROOT.parent / name
        if old.exists():
            print(f'Previous directory detected, preserved: {old}')
    os.environ['GRADIO_TEMP_DIR'] = str(TEMP / 'gradio')
    os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'


def detect_hardware():
    import torch
    disk = shutil.disk_usage(ROOT)
    available = torch.cuda.is_available()
    gpus = [torch.cuda.get_device_properties(i)
            for i in range(torch.cuda.device_count())] if available else []
    names = [g.name for g in gpus]
    t4 = bool(names and 'T4' in names[0].upper())
    profile = ('DUAL T4' if t4 and len(names) >= 2 else 'T4') if t4 else (
        'P100' if names and 'P100' in names[0].upper() else 'CONSERVATIVE 16 GB')
    print('\nLILY WAN 2.2 STUDIO')
    print('GPU(s):', ', '.join(names) or 'NONE', f'({len(gpus)} visible)')
    print('VRAM:', ', '.join(f'GPU {i}: {g.total_memory / GIB:.2f} GiB'
                              for i, g in enumerate(gpus)) or 'NONE')
    print('PROFILE:', profile)
    print(f'FREE DISK: {disk.free / GIB:.2f} GiB / TOTAL: {disk.total / GIB:.2f} GiB')
    print('TORCH VERSION:', torch.__version__)
    print('CUDA VERSION:', torch.version.cuda, '| CUDA available:', available)
    if not available:
        raise StudioError('ENVIRONMENT', 'Enable a GPU accelerator in Kaggle Settings and Run again.')
    if gpus[0].total_memory < 14 * GIB:
        raise StudioError('ENVIRONMENT', 'This notebook requires a GPU with approximately 16 GB VRAM.')
    if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 4):
        raise StudioError('ENVIRONMENT', 'Torch 2.4 or newer is required. Select a current Kaggle image.')
    torch.cuda.set_device(0)
    # P100 can be visible but unusable with CUDA wheels that dropped sm_60.
    # Run real kernels, including FP8 storage conversion (not FP8 arithmetic).
    try:
        with torch.inference_mode():
            x = torch.ones((32, 32), device='cuda:0', dtype=torch.float16)
            assert torch.isfinite(x @ x).all().item()
            packed = x.to(torch.float8_e4m3fn)
            assert torch.isfinite(packed.float()).all().item()
            cube = torch.ones((1, 1, 3, 8, 8), device='cuda:0')
            assert torch.isfinite(torch.nn.functional.conv3d(
                cube, torch.ones((1, 1, 1, 3, 3), device='cuda:0'))).all().item()
            torch.cuda.synchronize()
            del x, packed, cube
    except Exception as exc:
        raise StudioError('ENVIRONMENT',
                          brief(exc) + '\nKaggle Torch cannot execute the required GPU operations. '
                          'Choose T4, or a Kaggle image with P100/sm_60 support. '
                          'This notebook will not replace Torch/CUDA.') from None
    torch.cuda.empty_cache()
    print('Generation uses GPU 0. Additional GPUs are intentionally unused.')
    return {'profile': profile, 't4': t4, 'names': names}


def protected_package(name):
    name = re.sub(r'[-_.]+', '-', name).lower()
    return (name in {'torch', 'torchvision', 'torchaudio', 'triton', 'pytorch-triton',
                     'numpy', 'scipy', 'pillow', 'ipykernel', 'ipython'}
            or name.startswith(('nvidia-', 'cuda-', 'cudnn', 'cupy')))


def prepare_environment():
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    names = {canonicalize_name(d.metadata['Name'])
             for d in metadata.distributions() if d.metadata.get('Name')}
    installed = {name: metadata.version(name) for name in names}
    # Broad supported ranges retain compatible Kaggle packages; exact versions
    # are used only for a missing/incompatible application dependency.
    dependencies = [
        ('gradio>=6.26,<7', 'gradio==6.26.0'),
        ('transformers>=5.16,<6', 'transformers==5.16.1'),
        ('huggingface-hub>=1.16,<2', 'huggingface-hub==1.30.0'),
        ('safetensors>=0.4.2', 'safetensors==0.8.0'),
        ('einops>=0.7', 'einops==0.8.1'),
        ('torchsde==0.2.6', 'torchsde==0.2.6'),
        ('sentencepiece>=0.2', 'sentencepiece==0.2.2'),
        ('av>=14.2,<19', 'av==18.1.0'),
        ('psutil>=5.9', 'psutil==7.2.2'),
        ('requests>=2.28', 'requests==2.34.2'),
        ('pyyaml>=6', 'pyyaml==6.0.3'),
    ]
    roots = []
    for compatible, pinned in dependencies:
        req = Requirement(compatible)
        have = installed.get(canonicalize_name(req.name))
        roots.append(compatible if have and req.specifier.contains(have) else pinned)
        print(f'  {req.name}: {have or "missing"}')
    # Freeze the GPU stack AND packages already imported by the notebook kernel.
    # An incompatible loaded package is an early error, never a forced restart.
    imported = {key.partition('.')[0] for key in sys.modules}
    loaded_distributions = {canonicalize_name(name)
                            for module, names in metadata.packages_distributions().items()
                            if module in imported for name in names}
    frozen = {n: v for n, v in installed.items()
              if protected_package(n) or n in loaded_distributions}
    for needed in ('torch', 'torchvision', 'numpy', 'scipy', 'pillow'):
        if needed not in installed:
            raise StudioError('ENVIRONMENT', f'Kaggle is missing {needed}. Select a standard GPU image.')
    constraints = TEMP / 'preserve-kaggle.txt'
    constraints.write_text(''.join(f'{n}=={v}\n' for n, v in sorted(frozen.items())))
    report = TEMP / 'install-plan.json'
    check_disk(1.5 * GIB)
    print('Checking dependency plan; existing Torch/CUDA and loaded packages are locked.')
    run_command([sys.executable, '-m', 'pip', 'install', '--dry-run', '--report', report,
                 '--only-binary=:all:', '--no-cache-dir', '--disable-pip-version-check',
                 '--constraint', constraints, *roots], timeout=240)
    plan = json.loads(report.read_text())['install']
    packages = []
    for entry in plan:
        name = canonicalize_name(entry['metadata']['name'])
        if protected_package(name) or name in loaded_distributions:
            raise StudioError('ENVIRONMENT', f'Installation tried to change protected package {name}. Nothing installed.')
        packages.append(f"{name}=={entry['metadata']['version']}")
    if len(packages) > 65:
        raise StudioError('ENVIRONMENT', 'Unexpectedly large dependency plan. Nothing installed.')
    if packages:
        print('Installing only audited application packages:', ', '.join(packages))
        # Actual installation has NO dependency resolution, so it cannot pull
        # any unreviewed Torch, CUDA, cuDNN, or NVIDIA package.
        run_command([sys.executable, '-m', 'pip', 'install', '--no-deps',
                     '--only-binary=:all:', '--no-cache-dir', '--disable-pip-version-check',
                     *packages], timeout=360)
        importlib.invalidate_caches()
    else:
        print('Compatible dependencies already present; no installation needed.')
    for name, version in frozen.items():
        if metadata.version(name) != version:
            raise StudioError('ENVIRONMENT', f'Protected package changed unexpectedly: {name}')
    import gradio
    import numpy
    import torchvision
    import av
    from PIL import Image, ImageOps
    from safetensors import safe_open
    print('UI, image, tensor, and safetensors imports passed.')


def ensure_code():
    if not shutil.which('git'):
        raise StudioError('ENVIRONMENT', 'git is missing from the Kaggle image.')
    if not CODE.exists():
        staging = ROOT / 'ComfyUI.partial'
        if staging.exists():
            shutil.rmtree(staging)  # Only this notebook's incomplete source clone.
        print('Fetching pinned native inference core (ComfyUI v0.3.59).')
        run_command(['git', 'clone', '--depth', '1', '--branch', 'v0.3.59',
                     'https://github.com/Comfy-Org/ComfyUI.git', staging], timeout=180)
        staging.rename(CODE)
    result = subprocess.run(['git', '-C', str(CODE), 'rev-parse', 'HEAD'],
                            capture_output=True, text=True, timeout=15)
    if result.returncode or result.stdout.strip() != COMFY_REVISION:
        raise StudioError('ENVIRONMENT', 'The native inference checkout does not match the pinned revision.')
    if str(CODE) not in sys.path:
        sys.path.insert(0, str(CODE))


def preflight_backend(hardware):
    if STATE.backend is not None:
        return STATE.backend
    import torch
    from comfy.cli_args import args
    # Configure the library before importing its model manager. Do not parse or
    # rewrite ipykernel's arguments and do not call ComfyUI main.py.
    args.lowvram = True
    args.disable_xformers = True
    args.fp16_unet = True
    args.fp32_vae = True
    args.use_pytorch_cross_attention = hardware['t4']
    args.use_split_cross_attention = not hardware['t4']
    args.force_upcast_attention = True
    args.reserve_vram = 2.0
    import comfy.sd as sd
    import comfy.sample as sample
    import comfy.model_management as mm
    import comfy.utils as utils
    from comfy_extras.nodes_wan import Wan22ImageToVideoLatent
    from comfy_extras.nodes_model_advanced import ModelSamplingSD3
    required = {'model', 'noise', 'steps', 'cfg', 'sampler_name', 'scheduler',
                'positive', 'negative', 'latent_image', 'noise_mask', 'callback', 'seed'}
    if not required.issubset(inspect.signature(sample.sample).parameters):
        raise StudioError('ENVIRONMENT', 'Native sampling API mismatch.')
    if not {'vae', 'width', 'height', 'length', 'batch_size', 'start_image'}.issubset(
            inspect.signature(Wan22ImageToVideoLatent.execute).parameters):
        raise StudioError('ENVIRONMENT', 'Wan 2.2 image-conditioning API mismatch.')
    if 'uni_pc' not in sample.comfy.samplers.KSampler.SAMPLERS:
        raise StudioError('ENVIRONMENT', 'The UniPC sampler is unavailable.')
    if 'simple' not in sample.comfy.samplers.KSampler.SCHEDULERS:
        raise StudioError('ENVIRONMENT', 'The simple scheduler is unavailable.')
    # Exercise the actual official node with a tiny fake VAE, before any weights.
    class ProbeVAE:
        def encode(self, image):
            return torch.ones((1, 48, 1, image.shape[1] // 16, image.shape[2] // 16))
    latent = Wan22ImageToVideoLatent.execute(
        ProbeVAE(), 64, 64, 5, 1, torch.zeros((1, 64, 64, 3)))[0]
    assert latent['samples'].shape == (1, 48, 2, 4, 4)
    assert latent['noise_mask'][:, :, 0].count_nonzero().item() == 0
    assert latent['noise_mask'][:, :, 1].min().item() == 1
    assert torch.equal(latent['samples'][:, :, 0], torch.ones((1, 48, 4, 4)))
    STATE.backend = SimpleNamespace(sd=sd, sample=sample, mm=mm, utils=utils,
                                    latent_node=Wan22ImageToVideoLatent,
                                    sampling_node=ModelSamplingSD3)
    print('Official image-conditioning node, sampler, and backend imports passed.')
    return STATE.backend


def file_path(spec):
    return MODELS / Path(spec['path']).name


def validate_model(path, spec):
    from safetensors import safe_open
    if path.stat().st_size != spec['size']:
        raise StudioError('MODEL VALIDATION', f'{path.name}: wrong file size.')
    stamp = MODELS / (path.name + '.verified.json')
    identity = {'size': spec['size'], 'mtime_ns': path.stat().st_mtime_ns,
                'sha256': spec['sha256']}
    try:
        cached = json.loads(stamp.read_text()) == identity
    except (OSError, ValueError):
        cached = False
    if not cached:
        print(f'Validating SHA-256: {path.name}', flush=True)
        digest = hashlib.sha256()
        with path.open('rb') as source:
            for chunk in iter(lambda: source.read(8 * 1024 ** 2), b''):
                digest.update(chunk)
        if digest.hexdigest() != spec['sha256']:
            # Remove only the bad canonical file/link; preserve external data.
            path.unlink()
            stamp.unlink(missing_ok=True)
            raise StudioError('MODEL VALIDATION', f'{path.name}: checksum mismatch; invalid copy removed. Run again.')
        try:
            with safe_open(str(path), framework='pt', device='cpu') as weights:
                if not list(weights.keys()):
                    raise ValueError('No tensors in file')
        except Exception as exc:
            raise StudioError('MODEL VALIDATION', f'{path.name}: {brief(exc)}') from None
        stamp.write_text(json.dumps(identity))


def model_metadata():
    import requests
    try:
        url = f'https://huggingface.co/api/models/{MODEL_REPO}/revision/{MODEL_REVISION}'
        response = requests.get(url, params={'blobs': 'true'}, timeout=(15, 60))
        response.raise_for_status()
        data = response.json()
        if data['sha'] != MODEL_REVISION:
            raise ValueError('Unexpected model revision')
        files = {item['rfilename']: item for item in data['siblings']}
        for spec in MODEL_FILES:
            item = files[spec['path']]
            if item['size'] != spec['size'] or item['lfs']['sha256'] != spec['sha256']:
                raise ValueError(f"Upstream file metadata mismatch: {spec['path']}")
        print('Model repository, revision, filenames, byte counts, and checksums verified.')
    except Exception as exc:
        raise StudioError('MODEL DOWNLOAD', 'Cannot verify upstream metadata. '
                          'Enable Internet and Run again. ' + brief(exc)) from None


def download_file(spec):
    import requests
    path = file_path(spec)
    part = path.with_suffix(path.suffix + '.part')
    url = f"https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{spec['path']}"
    for attempt in range(3):
        try:
            offset = part.stat().st_size if part.exists() else 0
            if offset > spec['size']:
                part.unlink()
                offset = 0
            if offset < spec['size']:
                print(f'Downloading {path.name}: {offset / GIB:.2f}/{spec["size"] / GIB:.2f} GiB', flush=True)
                headers = {'Range': f'bytes={offset}-', 'Accept-Encoding': 'identity'}
                # Fresh resolve request avoids expired cached CDN signatures.
                with requests.get(url, headers=headers, stream=True, timeout=(20, 90),
                                  params={'download': 'true', 'lily_request': str(time.time_ns())}) as response:
                    response.raise_for_status()
                    if response.status_code == 206:
                        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', response.headers.get('Content-Range', ''))
                        if not match or int(match[1]) != offset or int(match[3]) != spec['size']:
                            raise ValueError('Invalid resume Content-Range; partial file preserved')
                        mode = 'ab'
                    elif response.status_code == 200:
                        offset = 0  # Server ignored Range: truncate the same partial file.
                        mode = 'wb'
                    else:
                        raise ValueError(f'Unexpected HTTP status {response.status_code}')
                    last = time.monotonic()
                    with part.open(mode) as dest:
                        for chunk in response.iter_content(4 * 1024 ** 2):
                            if not chunk:
                                continue
                            if offset + len(chunk) > spec['size']:
                                raise ValueError('Response exceeds expected model size')
                            dest.write(chunk)
                            offset += len(chunk)
                            if time.monotonic() - last >= 10:
                                print(f'  {path.name}: {100 * offset / spec["size"]:.1f}% '
                                      f'({offset / GIB:.2f} GiB)', flush=True)
                                last = time.monotonic()
                if part.stat().st_size != spec['size']:
                    raise IOError('Download ended early; partial file preserved')
            part.replace(path)  # Atomic rename, never a second multi-GB copy.
            validate_model(path, spec)
            print(f'Complete: {path.name} | FREE DISK: {shutil.disk_usage(ROOT).free / GIB:.2f} GiB')
            return
        except StudioError:
            raise
        except OSError as exc:
            if getattr(exc, 'errno', None) == 28:
                raise StudioError('INSUFFICIENT DISK SPACE', 'Disk filled during download; partial file preserved.') from None
            last_error = brief(exc)
        except Exception as exc:
            last_error = brief(exc)
        if attempt < 2:
            print(f'Connection interrupted; resuming ({attempt + 2}/3). {last_error}')
            time.sleep(2 * (attempt + 1))
    raise StudioError('MODEL DOWNLOAD', last_error + '\nPartial download preserved. Run again to resume.')


def ensure_model():
    model_metadata()  # Small metadata request before any large downloads.
    for spec in MODEL_FILES:
        path = file_path(spec)
        if not path.exists():
            # Reuse exact known previous model locations without copying weights.
            previous = ROOT.parent / 'ComfyUI' / 'models' / Path(spec['path']).relative_to('split_files')
            if previous.is_file() and previous.stat().st_size == spec['size']:
                path.symlink_to(previous)
                print('Reusing previous weights:', previous)
        if path.exists() and path.stat().st_size == spec['size']:
            validate_model(path, spec)
        elif path.exists():
            if path.is_symlink():
                path.unlink()
            else:
                part = path.with_suffix(path.suffix + '.part')
                if not part.exists() or path.stat().st_size > part.stat().st_size:
                    path.replace(part)
                else:
                    path.unlink()
    remaining = 0
    for spec in MODEL_FILES:
        path = file_path(spec)
        part = path.with_suffix(path.suffix + '.part')
        if not path.exists():
            present = part.stat().st_size if part.exists() and part.stat().st_size <= spec['size'] else 0
            remaining += spec['size'] - present
    print(f'Model final disk usage: {sum(s["size"] for s in MODEL_FILES) / GIB:.2f} GiB')
    print(f'Remaining downloads: {remaining / GIB:.2f} GiB; reserve: 1.50 GiB')
    check_disk(remaining + 1.5 * GIB)
    for spec in MODEL_FILES:
        if not file_path(spec).exists():
            download_file(spec)
        else:
            print('Verified model reused:', file_path(spec).name)


def load_pipeline():
    if STATE.pipeline is not None:
        print('Reusing the already loaded inference components.')
        return STATE.pipeline
    import torch
    import psutil
    available = psutil.virtual_memory().available
    print(f'AVAILABLE SYSTEM RAM: {available / GIB:.1f} GiB')
    if available < 21 * GIB:
        raise StudioError('ENVIRONMENT', 'At least 21 GiB available system RAM is required for CPU offloading. '
                          'Start a fresh Kaggle GPU session with about 29 GB RAM.')
    backend = STATE.backend
    print('Loading Wan 2.2 5B transformer with CPU offloading...')
    model = backend.sd.load_diffusion_model(str(file_path(MODEL_FILES[0])),
                                           model_options={'dtype': torch.float16})
    print('Loading compact text encoder (FP8 storage, ordinary FP32 computation)...')
    clip = backend.sd.load_clip([str(file_path(MODEL_FILES[1]))], clip_type=backend.sd.CLIPType.WAN,
                                model_options={'initial_device': torch.device('cpu')})
    print('Loading Wan 2.2 VAE in FP32...')
    vae = backend.sd.VAE(sd=backend.utils.load_torch_file(str(file_path(MODEL_FILES[2])), safe_load=True),
                         device=torch.device('cuda:0'), dtype=torch.float32)
    vae.throw_exception_if_invalid()
    if model.model.latent_format.latent_channels != 48:
        raise StudioError('BACKEND STARTUP', 'Loaded model is not the Wan 2.2 5B 48-channel model.')
    STATE.pipeline = SimpleNamespace(model=model, clip=clip, vae=vae)
    return STATE.pipeline


def presets(hardware):
    # width, height, frames, steps. Square/portrait keep the same pixel budget.
    values = [(448, 256, 49, 12), (512, 288, 81, 20), (576, 320, 121, 30)] if hardware['t4'] else [
        (320, 192, 33, 12), (384, 224, 49, 20), (448, 256, 81, 30)]
    return {name: dict(zip(('width', 'height', 'frames', 'steps'), value))
            for name, value in zip(PRESET_NAMES, values)}


def generation_settings(preset, aspect, image):
    settings = presets(STATE.hardware)[preset].copy()
    width, height = settings['width'], settings['height']
    if aspect == 'Portrait':
        width, height = height, width
    elif aspect in ('Match image', 'Square'):
        ratio = max(0.5, min(2.0, image.width / image.height)) if aspect == 'Match image' else 1.0
        area = width * height
        width = max(64, int(math.sqrt(area * ratio)) // 32 * 32)
        height = max(64, int(math.sqrt(area / ratio)) // 32 * 32)
    settings.update(width=width, height=height)
    return settings


def lower_memory(settings):
    return {**settings, 'width': max(64, int(settings['width'] * 0.75) // 32 * 32),
            'height': max(64, int(settings['height'] * 0.75) // 32 * 32),
            'frames': min(33, settings['frames'])}


def clear_gpu():
    import torch
    if STATE.backend:
        STATE.backend.mm.unload_all_models()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def is_cuda_oom(exc):
    import torch
    return isinstance(exc, torch.cuda.OutOfMemoryError) or (
        'out of memory' in str(exc).lower() and 'cuda' in str(exc).lower())


def infer_frames(image, prompt, negative, seed, settings, report):
    import numpy as np
    import torch
    from PIL import Image, ImageOps
    backend, pipeline = STATE.backend, STATE.pipeline
    with torch.inference_mode():
        report('Encoding motion prompt')
        key = (prompt, negative)
        if STATE.prompt_cache is None or STATE.prompt_cache[0] != key:
            positive = pipeline.clip.encode_from_tokens_scheduled(pipeline.clip.tokenize(prompt))
            negative_condition = pipeline.clip.encode_from_tokens_scheduled(pipeline.clip.tokenize(negative))
            if not all(torch.isfinite(c[0]).all().item() for c in positive + negative_condition):
                raise StudioError('GENERATION', 'Text encoder produced non-finite values.')
            STATE.prompt_cache = (key, positive, negative_condition)
        _, positive, negative_condition = STATE.prompt_cache
        report('Encoding starting image')
        resized = ImageOps.fit(image, (settings['width'], settings['height']), method=Image.Resampling.LANCZOS)
        pixels = torch.from_numpy(np.array(resized, dtype=np.float32) / 255.0).unsqueeze(0)
        latent = backend.latent_node.execute(pipeline.vae, settings['width'], settings['height'],
                                              settings['frames'], 1, pixels)[0]
        noise = backend.sample.prepare_noise(latent['samples'], seed)
        # Official model-sampling node; shift 5 suits these reduced resolutions.
        model = backend.sampling_node().patch(pipeline.model, shift=5.0)[0]
        def on_step(step, x0, x, total_steps):
            report(f'Denoising {step + 1}/{total_steps}', (step + 1) / total_steps)
        sampled = backend.sample.sample(model, noise, settings['steps'], 5.0, 'uni_pc', 'simple',
                                        positive, negative_condition, latent['samples'],
                                        noise_mask=latent['noise_mask'], callback=on_step,
                                        disable_pbar=True, seed=seed)
        if not torch.isfinite(sampled).all().item():
            raise StudioError('GENERATION', 'Denoising produced non-finite values; try TURBO with a new seed.')
        report('Decoding video in small tiles')
        decoded = pipeline.vae.decode_tiled(sampled, tile_x=16, tile_y=16, overlap=4,
                                          tile_t=8, overlap_t=1)
        decoded = decoded.reshape(-1, decoded.shape[-3], decoded.shape[-2], decoded.shape[-1])
        if decoded.shape[0] != settings['frames']:
            raise StudioError('GENERATION', 'Decoded frame count did not match the requested duration.')
        # Move output off the GPU and release denoiser/VAE residency between jobs.
        return decoded.cpu()


def encode_video(frames, destination, fps=FPS):
    import numpy as np
    destination = Path(destination)
    temporary = TEMP / f'encode_{secrets.token_hex(8)}.part.mp4'
    log_path = TEMP / 'ffmpeg.log'
    height, width = frames.shape[1:3]
    command = ['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
               '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s:v', f'{width}x{height}',
               '-r', str(fps), '-i', 'pipe:0', '-an', '-c:v', 'libx264',
               '-threads', '2', '-preset', 'veryfast', '-crf', '20',
               '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(temporary)]
    process = None
    try:
        with log_path.open('w') as log:
            process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=log)
            for frame in frames:
                array = np.asarray(frame)
                if not np.isfinite(array).all():
                    raise ValueError('Non-finite decoded pixels')
                rgb = np.ascontiguousarray(np.rint(np.clip(array, 0, 1) * 255), dtype=np.uint8)
                process.stdin.write(rgb.tobytes())
            process.stdin.close()
            if process.wait(timeout=120) != 0:
                raise RuntimeError('FFmpeg returned an error')
        if not temporary.is_file() or temporary.stat().st_size < 100:
            raise RuntimeError('FFmpeg did not create a valid MP4')
        temporary.replace(destination)
    except Exception as exc:
        if process and process.poll() is None:
            process.kill()
            process.wait(timeout=10)
        tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-6:]) if log_path.exists() else ''
        raise StudioError('VIDEO ENCODE', brief(exc) + '\n' + tail) from None
    finally:
        if process and process.stdin and not process.stdin.closed:
            try:
                process.stdin.close()
            except OSError:
                pass
        temporary.unlink(missing_ok=True)


def preflight_video():
    import numpy as np
    if not shutil.which('ffmpeg'):
        raise StudioError('ENVIRONMENT', 'FFmpeg is missing from the Kaggle image.')
    probe = TEMP / 'encode-probe.mp4'
    try:
        encode_video(np.zeros((2, 32, 32, 3), dtype=np.float32), probe)
    finally:
        probe.unlink(missing_ok=True)
    print('H.264 / yuv420p / faststart encoding smoke test passed.')


def generate_video(image, prompt, negative, preset, aspect, seed, randomize, progress=None):
    import gradio as gr
    from PIL import ImageOps
    if progress is None:
        progress = gr.Progress()
    if not STATE.lock.acquire(blocking=False):
        raise gr.Error('A generation or setup is already running.')
    started = time.monotonic()
    used_seed = None
    settings = None
    frames = None
    try:
        if image is None or not (prompt or '').strip():
            raise StudioError('GENERATION', 'Upload an image and enter a motion prompt.')
        if len(prompt) > 2000 or len(negative or '') > 2000:
            raise StudioError('GENERATION', 'Keep each prompt under 2,000 characters.')
        image = ImageOps.exif_transpose(image).convert('RGB')
        used_seed = secrets.randbelow(2 ** 32) if randomize else int(seed)
        if not 0 <= used_seed < 2 ** 32:
            raise StudioError('GENERATION', 'Use a seed from 0 to 4294967295.')
        settings = generation_settings(preset, aspect, image)
        check_disk(0.5 * GIB)
        def report(stage, fraction=None):
            elapsed = time.monotonic() - started
            desc = (f'{stage} | {preset} | {settings["width"]}×{settings["height"]}, '
                    f'{settings["frames"]} frames | seed {used_seed} | {elapsed:.0f}s')
            progress(fraction, desc=desc)
            print(desc, flush=True)
        retried = False
        for attempt in range(2):
            retry = False
            try:
                frames = infer_frames(image, prompt.strip(), negative or '', used_seed, settings, report)
                break
            except Exception as exc:
                if not is_cuda_oom(exc):
                    raise
                if attempt:
                    raise StudioError('CUDA OUT OF MEMORY', 'Both attempts ran out of VRAM. '
                                      'Select TURBO; start a fresh GPU session if other notebooks hold memory.') from None
                retry = True
            # Leave the exception scope before collecting: no traceback retains
            # tensors from the failed attempt while the retry starts.
            if retry:
                clear_gpu()
                settings = lower_memory(settings)
                retried = True
                report('VRAM exhausted; retrying once with fewer pixels and frames')
        report('Encoding iPhone-compatible MP4')
        destination = OUTPUTS / f'lily_{used_seed}_{secrets.token_hex(4)}.mp4'
        encode_video(frames, destination)
        elapsed = time.monotonic() - started
        status = (f'DONE · {STATE.hardware["profile"]} · {preset} · {elapsed:.1f}s · seed {used_seed}\n'
                  f'{settings["width"]}×{settings["height"]} · {settings["frames"]} frames '
                  f'· {settings["frames"] / FPS:.2f}s at {FPS} fps · {settings["steps"]} steps')
        if retried:
            status += '\nUsed the automatic lower-memory retry.'
        progress(1, desc='Done')
        return str(destination), str(destination), status, used_seed
    except Exception as exc:
        message = str(exc) if isinstance(exc, StudioError) else f'FAILED: GENERATION\n{brief(exc)}'
        message += f'\nElapsed: {time.monotonic() - started:.1f}s · seed: {used_seed}'
        print(message, flush=True)
        return None, None, message, used_seed
    finally:
        frames = None
        try:
            clear_gpu()
        except Exception as exc:
            print('GPU cleanup:', brief(exc))
        finally:
            STATE.lock.release()


def launch_ui():
    import gradio as gr
    if STATE.demo is not None:
        STATE.demo.close()
    css = '.gradio-container{max-width:680px!important;margin:auto!important} button{min-height:48px}'
    block_options = {'title': 'Lily Wan 2.2 Studio', 'delete_cache': (3600, 86400)}
    with gr.Blocks(**block_options) as demo:
        gr.Markdown(f'# Lily Wan 2.2 Studio\n**{STATE.hardware["profile"]} · GPU 0 · Wan 2.2 5B**')
        image = gr.Image(label='Starting image', type='pil', sources=['upload'])
        prompt = gr.Textbox(label='Motion prompt', lines=3, placeholder='Describe the movement you want to see.')
        preset = gr.Radio(PRESET_NAMES, value=PRESET_NAMES[0], label='Quality')
        aspect = gr.Dropdown(['Match image', 'Portrait', 'Landscape', 'Square'], value='Match image', label='Shape')
        with gr.Accordion('Seed and negative prompt', open=False):
            randomize = gr.Checkbox(value=True, label='Randomize seed')
            seed = gr.Number(value=42, precision=0, label='Seed')
            negative = gr.Textbox(value=DEFAULT_NEGATIVE, label='Negative prompt (optional)', lines=2)
        button = gr.Button('Generate video', variant='primary', size='lg')
        video = gr.Video(label='Your video', format='mp4')
        download = gr.File(label='Save MP4', interactive=False)
        status = gr.Textbox(label='Status', value='Ready. Start with TURBO.', lines=3, interactive=False)
        used_seed = gr.Number(label='Used seed', precision=0, interactive=False)
        # Use the real Gradio Progress default so queue progress reaches Safari.
        def generate(image, prompt, negative, preset, aspect, seed, randomize, progress=gr.Progress()):
            return generate_video(image, prompt, negative, preset, aspect, seed, randomize, progress)
        button.click(generate, [image, prompt, negative, preset, aspect, seed, randomize],
                     [video, download, status, used_seed], concurrency_limit=1, api_name=False)
        gr.Markdown('Keep the Kaggle session running. Shapes may center-crop the image. '
                    'Generation can take many minutes; the progress panel shows each step.')
    demo.queue(max_size=2, default_concurrency_limit=1)
    options = dict(share=True, server_name='0.0.0.0', inline=False, css=css,
                   prevent_thread_lock=True, show_error=False, allowed_paths=[str(OUTPUTS)])
    STATE.demo = demo  # Retain even on tunnel failure so rerun can close it.
    demo.launch(**options)
    if not demo.share_url:
        raise StudioError('BACKEND STARTUP', 'The Gradio share tunnel did not open. '
                          'Internet must be on. Run the cell again; models will be reused.')
    print('[5/5] READY')
    print('Open on iPhone:', demo.share_url)
    print('MP4 output directory:', OUTPUTS)
    return demo


def main():
    if not STATE.lock.acquire(blocking=False):
        print('A generation or setup is already running. Wait for it to finish.')
        return
    category = 'ENVIRONMENT'
    try:
        print('[1/5] ENVIRONMENT', flush=True)
        prepare_directories()
        STATE.hardware = detect_hardware()
        prepare_environment()
        ensure_code()
        preflight_backend(STATE.hardware)
        preflight_video()
        # Host-RAM check also precedes the weight download, not only model load.
        import psutil
        if STATE.pipeline is None and psutil.virtual_memory().available < 21 * GIB:
            raise StudioError('ENVIRONMENT', 'Need 21 GiB available system RAM for offloading. '
                              'Use a fresh Kaggle GPU session with about 29 GB RAM.')
        print('[2/5] MODEL', flush=True)
        category = 'MODEL DOWNLOAD'
        ensure_model()
        print('[3/5] BACKEND', flush=True)
        category = 'BACKEND STARTUP'
        load_pipeline()
        print('[4/5] UI', flush=True)
        launch_ui()
    except Exception as exc:
        if isinstance(exc, StudioError):
            print(str(exc), flush=True)
        else:
            print(f'FAILED: {category}\n{brief(exc)}', flush=True)
        if STATE.pipeline is None:
            gc.collect()
        print('Correct the reported Kaggle setting or connection, then Run this same cell again.')
    finally:
        STATE.lock.release()


main()
